# Experiment 9 – Association Rule Mining using Apriori
**Dataset:** Food_Delivery_Customer_Churn.csv (500 records)

**Aim:** Find frequent itemsets and association rules from the categorical attributes using the Apriori algorithm (mlxtend).

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from mlxtend.frequent_patterns import apriori, association_rules

df = pd.read_csv('Food_Delivery_Customer_Churn.csv')
print(df.shape)
df.head()

(500, 11)


,Age,City,Subscription_Type,Orders_Per_Month,Average_Order_Value,Average_Delivery_Time,Discount_Usage,Favorite_Category,Device,Tenure_Months,Churn
0,56,Thane,Free,2,151,36,39,Indian,Android,4,Retained
1,46,Mumbai,Free,8,738,42,13,Chinese,Web,11,Retained
2,32,Pune,Free,13,793,21,34,Indian,iOS,31,Churned
3,60,Navi Mumbai,Premium,13,239,49,47,Mexican,Android,43,Retained
4,25,Navi Mumbai,Free,1,499,69,83,Burgers,iOS,32,Churned


## Step 1 – Select categorical attributes and one-hot encode (each `attribute=value` becomes an item)

In [2]:
cat_cols = ['City','Subscription_Type','Favorite_Category','Device','Churn']
basket = pd.get_dummies(df[cat_cols].astype(str), prefix_sep='=').astype(bool)
print(basket.shape)
basket.head()

(500, 16)


,City=Mumbai,City=Navi Mumbai,City=Pune,City=Thane,Subscription_Type=Free,Subscription_Type=Premium,Favorite_Category=Burgers,Favorite_Category=Chinese,Favorite_Category=Indian,Favorite_Category=Mexican,Favorite_Category=Pizza,Device=Android,Device=Web,Device=iOS,Churn=Churned,Churn=Retained
0,False,False,False,True,True,False,False,False,True,False,False,True,False,False,False,True
1,True,False,False,False,True,False,False,True,False,False,False,False,True,False,False,True
2,False,False,True,False,True,False,False,False,True,False,False,False,False,True,True,False
3,False,True,False,False,False,True,False,False,False,True,False,True,False,False,False,True
4,False,True,False,False,True,False,True,False,False,False,False,False,False,True,True,False


## Step 2 – Frequent itemsets (min support = 0.10)

In [3]:
frequent_itemsets = apriori(basket, min_support=0.10, use_colnames=True)
frequent_itemsets['length'] = frequent_itemsets['itemsets'].apply(len)
frequent_itemsets = frequent_itemsets.sort_values('support', ascending=False).reset_index(drop=True)
print('Total frequent itemsets:', len(frequent_itemsets))
frequent_itemsets.head(15)

Total frequent itemsets: 61


,support,itemsets,length
0,0.622,frozenset({Subscription_Type=Free}),1
1,0.618,frozenset({Churn=Retained}),1
2,0.382,frozenset({Churn=Churned}),1
3,0.378,frozenset({Subscription_Type=Premium}),1
4,0.360,"frozenset({Churn=Retained, Subscription_Type=F...",2
5,0.342,frozenset({Device=Android}),1
6,0.330,frozenset({Device=iOS}),1
7,0.328,frozenset({Device=Web}),1
8,0.290,frozenset({City=Mumbai}),1
9,0.262,"frozenset({Churn=Churned, Subscription_Type=Fr...",2


## Step 3 – Association rules (min confidence = 0.60)

In [4]:
rules = association_rules(frequent_itemsets, metric='confidence', min_threshold=0.60)
rules = rules.sort_values(['lift','confidence'], ascending=False).reset_index(drop=True)
print('Total rules:', len(rules))
rules[['antecedents','consequents','support','confidence','lift']].head(15)

Total rules: 25


,antecedents,consequents,support,confidence,lift
0,"frozenset({Device=Android, Subscription_Type=P...",frozenset({Churn=Retained}),0.102,0.728571,1.178918
1,"frozenset({Churn=Churned, Device=Web})",frozenset({Subscription_Type=Free}),0.106,0.726027,1.167247
2,frozenset({Subscription_Type=Premium}),frozenset({Churn=Retained}),0.258,0.682540,1.104433
3,frozenset({Churn=Churned}),frozenset({Subscription_Type=Free}),0.262,0.685864,1.102675
4,"frozenset({Subscription_Type=Free, Device=iOS})",frozenset({Churn=Retained}),0.140,0.679612,1.099695
5,frozenset({Device=iOS}),frozenset({Churn=Retained}),0.222,0.672727,1.088555
6,frozenset({City=Pune}),frozenset({Subscription_Type=Free}),0.164,0.656000,1.054662
7,frozenset({Device=Web}),frozenset({Subscription_Type=Free}),0.214,0.652439,1.048937
8,frozenset({Favorite_Category=Indian}),frozenset({Subscription_Type=Free}),0.144,0.642857,1.033532
9,frozenset({Favorite_Category=Mexican}),frozenset({Churn=Retained}),0.130,0.637255,1.031157


## Step 4 – Rules with Churn as consequent

In [5]:
churn_rules = rules[rules['consequents'].apply(lambda s: any(i.startswith('Churn=') for i in s))]
print('Rules predicting Churn status:', len(churn_rules))
churn_rules[['antecedents','consequents','support','confidence','lift']].head(10)

Rules predicting Churn status: 13


,antecedents,consequents,support,confidence,lift
0,"frozenset({Device=Android, Subscription_Type=P...",frozenset({Churn=Retained}),0.102,0.728571,1.178918
2,frozenset({Subscription_Type=Premium}),frozenset({Churn=Retained}),0.258,0.682540,1.104433
4,"frozenset({Subscription_Type=Free, Device=iOS})",frozenset({Churn=Retained}),0.140,0.679612,1.099695
5,frozenset({Device=iOS}),frozenset({Churn=Retained}),0.222,0.672727,1.088555
9,frozenset({Favorite_Category=Mexican}),frozenset({Churn=Retained}),0.130,0.637255,1.031157
10,frozenset({City=Pune}),frozenset({Churn=Retained}),0.158,0.632000,1.022654
13,frozenset({City=Mumbai}),frozenset({Churn=Retained}),0.182,0.627586,1.015512
15,frozenset({Device=Android}),frozenset({Churn=Retained}),0.214,0.625731,1.012510
17,frozenset({Favorite_Category=Pizza}),frozenset({Churn=Retained}),0.120,0.625000,1.011327
18,frozenset({City=Thane}),frozenset({Churn=Retained}),0.152,0.622951,1.008011


## Step 5 – Visualizations & export

In [6]:
top = frequent_itemsets.head(10).copy()
top['label'] = top['itemsets'].apply(lambda s: ', '.join(sorted(s)))
plt.figure(figsize=(9,5))
plt.barh(top['label'][::-1], top['support'][::-1], color='steelblue')
plt.xlabel('Support'); plt.title('Top 10 Frequent Itemsets')
plt.tight_layout(); plt.savefig('frequent_itemsets_graph.png', dpi=150); plt.show()

/var/folders/_h/zjq63g2s5hn0k57qs774fdbh0000gn/T/ipykernel_94163/1111304670.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig('frequent_itemsets_graph.png', dpi=150); plt.show()


In [7]:
plt.figure(figsize=(7,5))
sc = plt.scatter(rules['support'], rules['confidence'], c=rules['lift'], cmap='viridis', s=40)
plt.colorbar(sc, label='Lift'); plt.xlabel('Support'); plt.ylabel('Confidence')
plt.title('Association Rules: Support vs Confidence'); plt.tight_layout()
plt.savefig('Experiment9_Graphs.png', dpi=150); plt.show()

/var/folders/_h/zjq63g2s5hn0k57qs774fdbh0000gn/T/ipykernel_94163/3242950618.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.savefig('Experiment9_Graphs.png', dpi=150); plt.show()


In [8]:
fi = frequent_itemsets.copy(); fi['itemsets'] = fi['itemsets'].apply(lambda s: ', '.join(sorted(s)))
fi.to_csv('frequent_itemsets.csv', index=False)
r = rules.copy()
for col in ['antecedents','consequents']: r[col] = r[col].apply(lambda s: ', '.join(sorted(s)))
r.to_csv('association_rules.csv', index=False)
print('Exported frequent_itemsets.csv and association_rules.csv')

Exported frequent_itemsets.csv and association_rules.csv
